# Introduction

This is a preliminary tutorial for the preliminary release of the SBI MRI package. Please do not use it for any official purposes yet, as it still needs to be validated. However, several quality of life improvements are there that you can use to speed up current implementations.

Importantly the package is split into two fundamental parts:
 - the models file
 - the SBI file

The models file is where most of the new stuff is - so I would be quite careful there. The SBI part is relatively standard, including on GPU speed up etc. So if you want to use one part of this package please start with that first.

# Models

In [ ]:
from Models import *


%load_ext autoreload

%autoreload 2

from dipy.data import get_fnames
from dipy.core.gradients import gradient_table
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.sphere import disperse_charges, Sphere, HemiSphere
from dipy.io.image import load_nifti

import requests
from pathlib import Path
import zipfile
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

This part of the package was built to allow for modular construction of dwMRI models. It means you instantiate a model and then you can add compartments to it as you see fit. Lets start by defining a random gradient table. This is taking directly from dipy and is a overview of your acquisition. If you are working multi diffusion time it is quite important to have that in there as follows:

In [ ]:
 # Just an example set 
fimg_init, fbvals, fbvecs = get_fnames(name='small_64D')
bvals, bvecs = read_bvals_bvecs(fbvals, fbvecs) 
hsph_initial = HemiSphere(xyz=bvecs[1:])
hsph_updated,_ = disperse_charges(hsph_initial,5000)
bvecs = np.vstack([[0,0,0],hsph_updated.vertices])
bvalsExt = np.hstack([bvals, 3000*np.ones_like(bvals)])
bvecsExt = np.vstack([bvecs, bvecs])
bvalsExt[65] = 0

In [ ]:
gtabSim = gradient_table(bvals=bvalsExt, bvecs=bvecsExt,small_delta = 0.007*np.ones_like(bvalsExt), big_delta = 0.0173*np.ones_like(bvalsExt))

Note please the: ```small_delta = 0.007*np.ones_like(bvalsExt), big_delta = 0.0173*np.ones_like(bvalsExt)```
Here we have all the same $\delta$ and $\Delta$, but if you have changing deltas please make sure this is reflected in the vector.

Then we can define a model as follows

In [ ]:
DTIModel = Model(gtabSim)
DTIModel._add_compartment("DTI","Hindered") # The first "DTI" is what kind of model we want and "Hindered" is the name we give it
DTIModel._set_default_SNR(20)

# or

AxCaliberModel = Model(gtabSim)
AxCaliberModel._add_compartment("DTI","Hindered")
AxCaliberModel._add_compartment("Sticks","Axons",Size='Infer')
AxCaliberModel._add_compartment("Sticks","Dendrites",Dispersion=True,Size='Infer')
AxCaliberModel._set_default_SNR(50)

# or

StickAndBallModel = Model(gtabSim)
StickAndBallModel._add_compartment("DTI","Hindered")
StickAndBallModel._add_compartment("Sticks","Axons",Size='Infer')
StickAndBallModel._add_compartment("Balls","Astrocytes")
StickAndBallModel._set_default_SNR(50)

# or 

StickAndBallFWModel = Model(gtabSim)
StickAndBallFWModel._add_compartment("DTI","Hindered")
StickAndBallFWModel._add_compartment("Sticks","Axons",Size='Infer')
StickAndBallFWModel._add_compartment("Balls","Astrocytes")
StickAndBallFWModel._add_compartment("FreeWater","Free Water")
StickAndBallFWModel._set_default_SNR(50)

All of the compartments have slightly arguments. If you want to figure out what these are you can do:

In [ ]:
help(Sticks)

If you then want simulate these models its as easy as:

In [ ]:
Params,Signals = StickAndBallFWModel.sample_and_simulation(10_000,parallel=True)

Conveniently, the models build your parameter list dynamically and can be accessed as:

In [ ]:
StickAndBallFWModel._get_parameter_names

For memory purposes its best to always use parallel = True - but currently the n_jobs is set to -1. So if you want to run on the cluster and dont mind waiting a bit you can set parallel to False.

Now you might have noticed that a file was saved - this is saved as DSBFW_todaysdat_timenow.h5. The DSBFW describes the model you are running and changes based on the compartments you have: 
- D = DTI
- S = Sticks
- B = Balls
- FW = Freewater

This way you can generate simulations and they will be automatically saved for reuse. If you want to add a random seed, its as simple as  

In [ ]:
Params,Signals = StickAndBallFWModel.sample_and_simulation(10_000,parallel=True,save=True,rng = np.random.default_rng(42))

By accident, we now overwrote the previous simulations - if we want to get them back there is a useful little function:

In [ ]:
P,S,S_raw,names,snr = Helpers.read_h5('saved_data/DSBFW_20260821_1104.h5') #(put the name of the file from before

Also right now the snr is set to a default 50 for the model (._set_default_SNR(50)). 
if you want to change it just put in: 

In [ ]:
Params,Signals = StickAndBallFWModel.sample_and_simulation(10_000,parallel=True,rng = np.random.default_rng(42),custom_snr=30)

# SBI

The SBI part will be something all of you are more familiar with. All i have done is put things in one place and sped some other things up. So how does this work in our current set-up.

We have generated some parameters and associated signals S. First we need to make sure that our data is in the right format, and that we don't include redundant information

In [ ]:
import Helpers #Custom functions
import SBI

In [ ]:
Par,Obs,Names,Bounds = Helpers.Prep_data(P,S,names)

Next we train the network

In [ ]:
Helpers.make_data_folder('test')

In [ ]:
Network = SBI.Train_Network_gpu(Par,Obs) #this will automatically see if you have speed-up possibilities (either GPU or mac MPS)

Lets say now we want to do some inference on an in-silico dataset

In [ ]:
Params_test,Signals_test = StickAndBallFWModel.sample_and_simulation(500,parallel=True,rng = np.random.default_rng(2026),custom_snr=30)

In [ ]:
Par_t,Obs_t,_,_ = Helpers.Prep_data(Params_test,Signals_test,StickAndBallFWModel._get_parameter_names)

In [ ]:
Result = SBI.Infer(Network,Obs_t)

We can also run this over a whole Volume using ```InferFromVolume()```. Here the input is the network, the data (can be 2D or 3D) and an associated Mask. Depending on the size of the data you are passing this might take a bit.

# Example of in-vivo

Now lets run an in-vivo example - this data includes multiple b-values and diffusion times, so it is suitable for both DTI and more advanced multi-compartment models.

In [ ]:
def download_dropbox(url, output_path):
    # Force Dropbox to return the file rather than the preview page
    if "dl=0" in url:
        url = url.replace("dl=0", "dl=1")
    elif "dl=1" not in url:
        url += "&dl=1" if "?" in url else "?dl=1"

    output_path = Path(output_path)

    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return output_path

First we make sure to download the data and unzip the file into ./dat.

In [ ]:
url = "https://www.dropbox.com/scl/fi/xki53zdriavev4rt22wjz/data.zip?rlkey=ejfw5malh99l98sv747jg273c&dl=1"
download_dropbox(url, "data3.zip")
with zipfile.ZipFile("data3.zip", "r") as z:
    z.extractall("./dat")

In [ ]:
fdwi = './dat/output.nii.gz'
bvaloc = './dat/bvals.bvals'
bveloc = './dat/bvecs.bvecs'
deltas = './dat/deltas.deltas'
data, affine, img = load_nifti(fdwi, return_img=True)
bvals = np.loadtxt(bvaloc)
bvecs = np.loadtxt(bveloc)
deltas = np.loadtxt(deltas)

Now importantly, this data is *NOT* normalised - however the implementation assumes it is. Therefore, it is key that you make sure that you divide by teh average of the $b_0$ - if you only have one, just divide by that one.

In [ ]:
data_norm = data/data[...,bvals==0].mean(axis=-1)[...,None]

Lets define a model with the correct gradient table:

In [ ]:
gtab = gradient_table(bvals=bvals, bvecs=bvecs,small_delta = 0.007*np.ones_like(bvals), big_delta = deltas*1e-3)

First, lets just start with a DTI model - keeping in mind that multiple b-values and Deltas are not really optimal for DTI:

In [ ]:
DTIModel = Model(gtab)
DTIModel._add_compartment("DTI","DTI")
DTIModel._set_default_SNR(50)

In [ ]:
Params,Signals = DTIModel.sample_and_simulation(10_000,parallel=True,rng = np.random.default_rng(42))
Par,Obs,Names,Bounds = Helpers.Prep_data(Params,Signals,DTIModel.parameter_list)

In [ ]:
DTINetwork = SBI.Train_Network_gpu(Par,Obs) #this will automatically see if you have speed-up possibilities (either GPU or mac MPS)

Now to the inference - we have a function "Infer_from_volume" that takes in a 2D/3D entity and returns it to you in the same shape. Importantly, it requires 

In [ ]:
Result = SBI.Infer_from_volume(DTINetwork,data_norm,batch_size=32)

Now as this is a DTI network, we would probably be interested in MD and FA. However, at this point we only have the tensors. Therefore, we want to obtain this as well:

In [ ]:
MDFA,Evals,Evecs = DTIModel.compartments['DTI'].EvaluateTensor(Result)
MD = MDFA[...,0]
FA = MDFA[...,1]

In [ ]:
Helpers.Plot_parameter_map(MD[:,:,24],DTIModel,'MD',data_range = [0,5e-3])

In [ ]:
Helpers.Plot_parameter_map(FA,DTIModel,'FA',data_range = [0,1])

There is even a useful little widge that lets you scroll through the parameter:

In [ ]:
Helpers.Widget_parameter_map(FA,DTIModel,'FA',data_range=[0,1],view='sag')

Now lets do a stick and ball model:

In [ ]:
SBModel = Model(gtab)
SBModel._add_compartment("DTI","Hindered")
SBModel._add_compartment("Sticks","Axons")
SBModel._add_compartment("Balls","Astrocytes")
SBModel._set_default_SNR(50)

In [ ]:
Params,Signals = SBModel.sample_and_simulation(10_000,parallel=True,rng = np.random.default_rng(42))
Par,Obs,Names,Bounds = Helpers.Prep_data(Params,Signals,SBModel.parameter_list)

In [ ]:
SBNetwork = SBI.Train_Network_gpu(Par,Obs)
Result = SBI.Infer_from_volume(SBNetwork,data_norm,batch_size=32)

Given that we removed one of the fractions for the training, we can add it back so we have a complete picture of the model parameters. Keep in mind that Prep_data also removed this from the name space, so we need to add that back as well.

In [ ]:
Result_aug,Names = Helpers.Add_back_fraction(Result,Names,SBModel)

Lets do some more plotting!

In [ ]:
Helpers.Plot_parameter_map(Result_aug,SBModel,'Axons_f',data_range = [0,1])

We can also plot all parameters all at once. Either in all three axis:

In [ ]:
Helpers.Plot_parameter_maps(Result_aug,SBModel,Names,cmap='grey',save=True)

or just with one view:

In [ ]:
Helpers.Plot_parameter_maps(Result_aug[:,:,24],SBModel,Names,cmap='viridis',save=True)

Or we can use the widget again:

In [ ]:
Helpers.Widget_parameter_map(Result_aug,SBModel,'Axons_f',data_range=[0,1])

In [ ]:
Helpers.Widget_parameter_map(Result_aug,SBModel,'Axons_f',data_range=[0,1],view='sag')

# Validation

In [ ]:
import Validation

This package also includes a Validation suite to test some of the parameters for the SBI inference and training. At this point, we only have this validation for posterior samples (ParameterEval_PS) and number of training samples (ParameterEval_TS). But future editions might include more parameter validation

In [ ]:
DTIModel = Model(gtabSim)
DTIModel._add_compartment("DTI","Hindered") # The first "DTI" is what kind of model we want and "Hindered" is the name we give it
DTIModel._set_default_SNR(30)

In [ ]:
help(Validation.ParameterEval_TS)

In [ ]:
X = Validation.ParameterEval_PS(DTIModel,30,fidelity = 5)

Here we do a brief explanation of the figure that comes out of one of these parameter validations. 
- Point Error convergence (Top left): Using the mean of the distribution as the estimate of that parameter, we can compare this to the real ground trutch and see how it changes as we add more posterior samples.
- 95% Coverage (Top middle): A way to see if the posterior is well defined. Basically, it says if we sample from the posterior, the true value should fall within the 95% interval 95% of the time. Therefore, we have the gray area, all the points hshould ideally lie wihin it.
- Width (Top right): How broad is the 95% of the posterior as a fraction of the respective full parameter space - the smaller the better obviously. Important that this is checked with the coverage
- Signal RMSE (Bottom left): This shows the error of the reconstructed siganl  versus the true clean signal - idea of magnitude.
- Signal Corr (Bottom middle): This shows the correlation of the reconstructed siganl with the true clean signal - idea of directionality.

The code will try and tell you if things have "converged" - but this is heuristic at best. By marking points as green if, for example, the change between points is small enough or we are within the 95% coverage, we try to give an idea of which parameter you should choose. However, importantly I would not rely on this too much and instead look at directky.

In [ ]:
X = Validation.ParameterEval_TS(DTIModel,30,fidelity = 5)